# TP4 - Découverte de modèles open source avec Hugging Face dans Google Colab

Dans ce TP, vous allez découvrir comment **charger et utiliser des modèles open source** depuis la plateforme **Hugging Face**, directement dans **Google Colab**.

L'objectif n'est pas d'entraîner un modèle vous-même, mais de comprendre comment **réutiliser un modèle préentraîné** sur différentes tâches simples.

Vous allez apprendre à :

- préparer un environnement Colab pour utiliser des modèles ;
- comprendre le lien entre un modèle et une tâche ;
- tester plusieurs usages simples ;
- observer les entrées et les sorties ;
- développer un regard critique sur les résultats.

Le fil conducteur du TP est le suivant : **un modèle n'est pas magique**. Il produit des sorties souvent utiles, parfois impressionnantes, mais aussi parfois discutables, incomplètes ou maladroites.


## Consignes de travail

- Exécutez les cellules **dans l'ordre**.
- Prenez le temps de lire les cellules Markdown avant de compléter le code.
- Quand une question demande un commentaire, rédigez une **réponse courte, claire et en français**.
- Regardez attentivement le **format des sorties** produites par les modèles.
- Retenez qu'ici, l'objectif principal est de **comprendre l'usage raisonné** des modèles open source.


## Outils utilisés

Dans ce TP, nous allons utiliser principalement :

- **Google Colab** : pour exécuter le notebook dans le cloud ;
- **Hugging Face** : pour accéder à des modèles open source préentraînés ;
- **`transformers`** : la bibliothèque Python qui permet de charger facilement ces modèles ;
- **`pipeline(...)`** : une interface simple qui permet d'utiliser un modèle pour une tâche donnée sans entrer dans tous les détails techniques.

Dans un premier temps, nous allons volontairement rester à un niveau simple et pratique.


## Plan du TP

1. Mise en place de l'environnement
2. Comprendre ce qu'est un modèle Hugging Face
3. Première utilisation guidée d'un modèle
4. Test de plusieurs tâches
5. Analyse critique des sorties
6. Influence des paramètres
7. Exercices guidés sur des cas neutres
8. Partie autonome
9. Synthèse finale


---
## Partie 1 - Mise en place de l'environnement

Avant d'utiliser un modèle Hugging Face, il faut s'assurer que l'environnement Colab est correctement configuré.

Dans cette partie, vous allez :

- vérifier l'environnement d'exécution ;
- vérifier si un GPU est disponible ;
- installer les bibliothèques nécessaires ;
- vérifier que les imports fonctionnent.

Même si certaines tâches peuvent fonctionner sur CPU, le GPU rend souvent l'exécution plus rapide, surtout lorsque les modèles deviennent plus lourds.


### Point important pour Colab

Si vous êtes dans Google Colab, vérifiez si le GPU est activé :

1. menu **Exécution** ;
2. **Modifier le type d'exécution** ;
3. choisir **GPU** si possible.

Le TP peut parfois fonctionner sans GPU, mais ce sera souvent plus lent.


In [ ]:
# Exercice 1 - Vérifier l'environnement
import sys
import platform

print("Version de Python :", sys.version)
print("Plateforme :", platform.platform())
print("Système :", platform.system())
print("Processeur :", platform.processor())


In [ ]:
# Exercice 2 - Vérifier le GPU
import torch

gpu_disponible = torch.cuda.is_available()
print("GPU disponible :", gpu_disponible)

if gpu_disponible:
    print("Nom du GPU :", torch.cuda.get_device_name(0))
else:
    print("Aucun GPU détecté. Le notebook peut fonctionner, mais certaines étapes seront plus lentes.")


In [ ]:
# Exercice 3 - Installer les bibliothèques nécessaires
# A exécuter une seule fois dans Google Colab.
# Le -q rend l'installation plus discrète dans la sortie du notebook.
%pip install -q transformers accelerate sentencepiece


In [ ]:
# Exercice 4 - Vérifier les imports
from transformers import pipeline
import torch

device_index = 0 if torch.cuda.is_available() else -1
print("Les imports fonctionnent.")
print("device_index utilisé pour les pipelines :", device_index)


**Question 1** : À quoi servent `transformers`, `pipeline` et éventuellement le GPU dans ce TP ?

_Votre réponse ici._


---
## Partie 2 - Comprendre ce qu'est un modèle Hugging Face

Avant d'utiliser un modèle, il faut comprendre l'idée générale.

Un modèle Hugging Face est généralement un **modèle préentraîné**. Cela signifie qu'il a déjà appris à partir d'une grande quantité de données, avant même que vous l'utilisiez.

Ensuite, ce modèle est souvent associé à une **tâche** particulière, par exemple :

- génération de texte ;
- résumé ;
- classification de texte ;
- traduction.

L'idée importante est la suivante :

**on peut réutiliser un modèle déjà entraîné sans avoir à refaire soi-même tout l'entraînement**.


### Le lien entre modèle et tâche

Dans `transformers`, on peut souvent charger un modèle avec une ligne du type :

`pipeline("nom_de_tache", model="nom_du_modele")`

Cela veut dire que :

- on choisit une **tâche** ;
- on choisit un **modèle adapté** à cette tâche ;
- on applique ce modèle à une entrée.

Le même modèle n'est pas forcément adapté à toutes les tâches. Il faut donc apprendre à choisir un modèle cohérent avec l'usage voulu.


**Question 2** : Qu'est-ce qu'un modèle préentraîné ? Pourquoi est-ce intéressant dans un TP comme celui-ci ?

_Votre réponse ici._


**Question 3** : Donnez au moins trois tâches différentes que l'on peut réaliser avec des modèles open source sur Hugging Face.

_Votre réponse ici._


---
## Partie 3 - Première utilisation guidée d'un modèle

Nous allons commencer par une tâche simple et facile à interpréter : la **classification de sentiment**.

L'idée est de donner une phrase au modèle, puis de voir s'il la classe par exemple comme plutôt positive ou négative.

Cette première étape est utile parce que :

- le format d'entrée est simple ;
- le format de sortie est souvent facile à lire ;
- on comprend rapidement ce que fait le modèle.

Attention : beaucoup de modèles courants de sentiment sont surtout performants sur de l'anglais. Il faut donc garder cela en tête quand on interprète les résultats.


In [ ]:
# Exercice 5 - Charger un premier modèle de classification
# Ce pipeline est spécialisé dans la classification de sentiment en anglais.
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=device_index,
)

print("Pipeline de sentiment chargé.")


In [ ]:
# Exercice 6 - Première prédiction guidée
# On choisit une phrase simple en anglais car le modèle utilisé est anglophone.
phrase_positive = "I really enjoyed this course because it was clear and useful."
resultat_sentiment = sentiment_pipe(phrase_positive)

print("Phrase test :", phrase_positive)
print("Sortie complète :", resultat_sentiment)


In [ ]:
# Exercice 7 - Observer le format de la sortie
print("Type Python de la sortie :", type(resultat_sentiment))
print("Premier élément :", resultat_sentiment[0])
print("Clés présentes dans la réponse :", resultat_sentiment[0].keys())


**Question 4** : Que fait exactement ce premier modèle ? Quelle entrée reçoit-il et quel type de sortie produit-il ?

_Votre réponse ici._


---
## Partie 4 - Test de plusieurs tâches

Nous allons maintenant découvrir plusieurs usages possibles des modèles open source.

L'objectif n'est pas seulement d'obtenir des résultats, mais aussi de comparer :

- le type d'entrée attendu ;
- le type de sortie obtenu ;
- la logique de la tâche ;
- la pertinence du modèle choisi.

Dans cette partie, vous allez tester :

- la génération de texte ;
- le résumé ;
- la classification de texte.


### 4.1 - Génération de texte

Dans une tâche de génération, on donne au modèle un **début de texte** ou une consigne courte, puis il produit une continuation.

Ce type de tâche est souvent impressionnant, mais il faut rester prudent : une suite grammaticalement plausible n'est pas forcément une suite pertinente ou correcte.


In [ ]:
# Exercice 8 - Charger un modèle de génération
# distilgpt2 est une version plus légère de GPT-2, pratique pour un premier essai.
generation_pipe = pipeline(
    "text-generation",
    model="distilgpt2",
    device=device_index,
)

print("Pipeline de génération chargé.")


In [ ]:
# Exercice 9 - Première génération
prompt_generation = "Science helps us understand the world because"

generation_resultat = generation_pipe(
    prompt_generation,
    max_new_tokens=40,
    do_sample=True,
    temperature=0.8,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

print("Prompt :", prompt_generation)
print("Sortie générée :")
print(generation_resultat[0]["generated_text"])


### 4.2 - Résumé automatique

Dans une tâche de résumé, on donne un texte plus long au modèle, qui doit essayer d'en produire une version plus courte.

Un bon résumé ne doit pas seulement être plus court : il doit aussi garder les idées essentielles.


In [ ]:
# Exercice 10 - Charger un modèle de résumé
# Ce modèle est souvent utilisé pour faire des résumés en anglais.
summary_pipe = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    device=device_index,
)

print("Pipeline de résumé chargé.")


In [ ]:
# Exercice 11 - Tester le résumé
texte_resume = (
    "Artificial intelligence is used in many areas such as education, health, transport and science. "
    "It can help people analyse large amounts of data, automate repetitive tasks and build useful tools. "
    "However, AI systems can also make mistakes, reflect biases from data and produce convincing but imperfect answers. "
    "For this reason, users need to understand both the potential and the limits of these systems."
)

resume_resultat = summary_pipe(texte_resume, max_length=45, min_length=20, do_sample=False)

print("Texte source :")
print(texte_resume)
print("\nRésumé produit :")
print(resume_resultat[0]["summary_text"])


### 4.3 - Classification de texte

Vous avez déjà testé une première classification de sentiment. L'idée ici est de comparer plusieurs entrées et d'observer si la sortie change de manière cohérente.


In [ ]:
# Exercice 12 - Comparer plusieurs phrases en classification
phrases_test = [
    "This movie was excellent and full of energy.",
    "The service was terrible and I am disappointed.",
    "The product is acceptable, but it could be better.",
]

resultats_phrases = sentiment_pipe(phrases_test)

for phrase, resultat in zip(phrases_test, resultats_phrases):
    print("Phrase :", phrase)
    print("Résultat :", resultat)
    print("-" * 60)


**Question 5** : Quelles différences voyez-vous entre les tâches de génération, de résumé et de classification ?

_Votre réponse ici._


---
## Partie 5 - Analyse critique des sorties

Maintenant que vous avez obtenu plusieurs sorties, il faut apprendre à les lire avec un peu de recul.

Un modèle peut produire une réponse :

- cohérente ;
- utile ;
- partiellement correcte ;
- vague ;
- discutable ;
- ou même trompeuse tout en ayant l'air plausible.

C'est une compétence importante : **ne pas confondre une sortie fluide avec une sortie fiable**.


In [ ]:
# Exercice 13 - Reprendre une sortie et l'analyser
# Ici, on reprend l'exemple du résumé pour l'observer plus facilement.
sortie_a_analyser = resume_resultat[0]["summary_text"]

print("Sortie choisie pour l'analyse :")
print(sortie_a_analyser)

print("\nPoint d'observation :")
print("La sortie est-elle fidèle au texte source ? Est-elle claire ? Oublie-t-elle des idées importantes ?")


**Question 6** : Donnez un exemple de sortie qui vous semble convaincante, et expliquez pourquoi.

_Votre réponse ici._


**Question 7** : Donnez un exemple de sortie qui vous semble discutable, imprécise ou limitée, et expliquez pourquoi.

_Votre réponse ici._


---
## Partie 6 - Influence des paramètres

Les sorties d'un modèle ne dépendent pas seulement de l'entrée. Elles dépendent aussi de certains **paramètres**.

Par exemple, en génération de texte, on peut modifier :

- la longueur maximale ;
- le caractère plus ou moins aléatoire de la génération ;
- la diversité des sorties.

L'objectif ici est de voir que deux exécutions avec la même idée de départ peuvent donner des résultats différents selon le paramétrage.


In [ ]:
# Exercice 14 - Modifier la longueur d'une génération
generation_courte = generation_pipe(
    prompt_generation,
    max_new_tokens=20,
    do_sample=True,
    temperature=0.8,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

generation_longue = generation_pipe(
    prompt_generation,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

print("Version courte :")
print(generation_courte[0]["generated_text"])
print("\nVersion longue :")
print(generation_longue[0]["generated_text"])


In [ ]:
# Exercice 15 - Jouer sur le caractère aléatoire
generation_deterministe = generation_pipe(
    prompt_generation,
    max_new_tokens=35,
    do_sample=False,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

generation_aleatoire = generation_pipe(
    prompt_generation,
    max_new_tokens=35,
    do_sample=True,
    temperature=1.0,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

print("Sortie avec do_sample=False :")
print(generation_deterministe[0]["generated_text"])
print("\nSortie avec do_sample=True :")
print(generation_aleatoire[0]["generated_text"])


**Question 8** : Quel effet avez-vous observé en modifiant les paramètres de génération ?

_Votre réponse ici._


---
## Partie 7 - Exercices guidés sur des cas neutres

Dans cette partie, vous allez réutiliser les modèles sur des exemples simples et généraux.

L'objectif est de gagner en autonomie tout en restant dans un cadre guidé.

Vous n'êtes pas obligé d'utiliser tous les cas proposés, mais vous devez en traiter plusieurs.


### Cas A - Résumer un texte court

Choisissez ou rédigez un petit texte informatif en anglais de quelques phrases. L'objectif est de voir si le modèle réussit à condenser l'idée principale.


In [ ]:
# Exercice 16 - Cas A : résumé
texte_info = (
    "Many schools now use digital tools to help students learn more actively. "
    "Teachers can share documents online, organize quizzes and give feedback more quickly. "
    "These tools can save time, but they also require training and reliable internet access."
)

resume_info = summary_pipe(texte_info, max_length=35, min_length=15, do_sample=False)

print("Texte source :")
print(texte_info)
print("\nRésumé :")
print(resume_info[0]["summary_text"])


### Cas B - Classer des avis

Prenez plusieurs phrases courtes exprimant des opinions différentes. L'objectif est de voir si le modèle classe ces avis de manière cohérente.


In [ ]:
# Exercice 17 - Cas B : classification d'avis
avis = [
    "I love this app, it is easy to use and very helpful.",
    "This was a bad experience and I want a refund.",
    "The idea is interesting, but the execution is average.",
    "The presentation was clear, useful and motivating.",
]

resultats_avis = sentiment_pipe(avis)

for avis_texte, resultat in zip(avis, resultats_avis):
    print("Avis :", avis_texte)
    print("Classification :", resultat)
    print("-" * 60)


### Cas C - Générer une courte suite à partir d'un prompt

L'idée ici est de voir si le modèle produit une suite grammaticalement correcte, cohérente et liée au prompt de départ.


In [ ]:
# Exercice 18 - Cas C : génération courte
prompt_court = "A good teacher helps students by"

generation_cas_c = generation_pipe(
    prompt_court,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.9,
    pad_token_id=generation_pipe.tokenizer.eos_token_id,
)

print("Prompt :", prompt_court)
print("\nTexte généré :")
print(generation_cas_c[0]["generated_text"])


### Cas D - Traduction simple (optionnel)

Si vous souhaitez tester une autre tâche, vous pouvez aussi essayer un modèle de traduction. Ce n'est pas obligatoire pour le TP, mais cela permet d'élargir la découverte.


In [ ]:
# Exercice 19 - Cas D : traduction optionnelle
# Cette partie reste optionnelle, mais le code est fourni pour faciliter l'essai.
translation_pipe = pipeline(
    "translation_fr_to_en",
    model="Helsinki-NLP/opus-mt-fr-en",
    device=device_index,
)

phrases_fr = [
    "Ce TP permet de découvrir les modèles open source.",
    "Les étudiants doivent analyser les limites des sorties produites.",
]

traductions = translation_pipe(phrases_fr)

for phrase_fr, traduction in zip(phrases_fr, traductions):
    print("Français :", phrase_fr)
    print("Anglais :", traduction["translation_text"])
    print("-" * 60)


**Question 9** : Parmi les exercices guidés, quel usage vous a semblé le plus convaincant ? Lequel vous a semblé le plus fragile ?

_Votre réponse ici._


---
## Partie 8 - Partie autonome

Vous allez maintenant réutiliser les outils de manière plus libre.

Choisissez **une tâche principale** parmi les propositions suivantes :

- résumé de texte ;
- classification de sentiment ou d'opinion ;
- reformulation ou génération courte ;
- comparaison de deux modèles sur une même tâche.

Votre travail devra montrer que vous êtes capable de :

- choisir un modèle adapté ;
- justifier ce choix ;
- tester plusieurs entrées ;
- comparer les résultats ;
- commenter la qualité des sorties.


### Étapes attendues dans la partie autonome

Voici une démarche conseillée :

1. choisissez une tâche ;
2. choisissez un modèle ;
3. expliquez brièvement votre choix ;
4. testez au moins deux ou trois entrées ;
5. comparez les sorties ;
6. rédigez un petit commentaire critique.


**Choix de la tâche autonome**

_Indiquez ici la tâche choisie._


In [ ]:
# Exercice 20 - Partie autonome : choix du modèle et premiers tests
# TODO 1 : chargez le modèle que vous avez choisi
# TODO 2 : préparez plusieurs entrées adaptées à votre tâche
# TODO 3 : exécutez le modèle sur ces entrées
# TODO 4 : affichez les résultats obtenus


**Question 10** : Quel modèle avez-vous choisi et pourquoi vous semble-t-il adapté à la tâche ?

_Votre réponse ici._


**Question 11** : Les différentes sorties obtenues sont-elles cohérentes entre elles ? Expliquez.

_Votre réponse ici._


**Question 12** : Quelles limites avez-vous observées dans votre partie autonome ?

_Votre réponse ici._


---
## Partie 9 - Synthèse finale

Cette dernière partie sert à reformuler clairement ce que vous avez appris.

L'objectif n'est pas d'écrire beaucoup, mais de montrer que vous avez compris :

- ce qu'est un modèle préentraîné ;
- comment le charger dans Colab ;
- comment l'utiliser avec une tâche précise ;
- comment interpréter ses sorties avec esprit critique.


**1. Quel modèle avez-vous utilisé dans la partie guidée ?**

_Votre réponse ici._


**2. Pour quelle tâche ou quelles tâches ?**

_Votre réponse ici._


**3. Quel type d'entrée avez-vous donné au modèle ?**

_Votre réponse ici._


**4. Quel type de sortie avez-vous obtenu ?**

_Votre réponse ici._


**5. Le résultat vous semble-t-il satisfaisant ? Pourquoi ?**

_Votre réponse ici._


**6. Quelles limites principales avez-vous observées ?**

_Votre réponse ici._


**7. Dans quel autre contexte pourrait-on utiliser un modèle du même type ?**

_Votre réponse ici._


## Fin du TP

Si vous avez terminé :

- relisez vos réponses ;
- vérifiez que les sorties générées sont bien visibles dans le notebook ;
- assurez-vous d'avoir testé plusieurs entrées ;
- gardez en tête qu'utiliser un modèle ne consiste pas seulement à obtenir une sortie, mais aussi à la **questionner** et à l'**interpréter avec prudence**.
